# Portfolio Optimization using Credit Risk Metrics

## Objective

This notebook builds an optimized lending portfolio using the outputs generated from the previous notebooks.

The optimization process combines:

- Probability of Default (PD)
- Loss Given Default (LGD)
- Exposure at Default (EAD)
- Expected Loss
- Expected Interest Income
- Risk-Adjusted Return
- Loan Decision

The objective is to identify borrowers that maximize expected portfolio profitability while maintaining acceptable credit risk.

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns',150)
pd.set_option('display.max_rows',150)

## Import Decision Engine Output

The borrower-level output generated by the decision engine is imported.

The dataset already contains:

- loan decisions
- PD scores
- LGD estimates
- EAD
- expected loss
- expected interest income
- risk-adjusted return
- borrower risk bands

These variables are used to construct an optimized lending portfolio.

In [3]:
df = pd.read_pickle("../data/decision_engine_output.pkl")
df.shape

(40000, 153)

## Review Key Portfolio Variables

The important borrower-level risk metrics are reviewed before portfolio optimization.

These include:

- loan decision
- probability of default
- expected loss
- expected interest income
- risk-adjusted return
- borrower risk band
- actual default flag

These metrics form the basis for portfolio selection.

In [4]:
df[["loan_decision","pd_score","lgd_estimate","ead","expected_loss","expected_interest_income",
    "risk_adjusted_return", "risk_band", "actual_default_flag"]].head()

,loan_decision,pd_score,lgd_estimate,ead,expected_loss,expected_interest_income,risk_adjusted_return,risk_band,actual_default_flag
0,Reject,0.495415,0.695648,12000,4135.613883,2386.8000,-1748.813883,High,1
1,Reject,0.504460,0.611491,20100,6200.299219,3133.5900,-3066.709219,High,0
2,Reject,0.301881,0.558952,8000,1349.895463,891.2000,-458.695463,Low,0
3,Reject,0.685799,0.611491,23325,9781.568456,2959.9425,-6821.625956,Very High,0
4,Reject,0.739917,0.716095,21250,11259.330155,4876.8750,-6382.455155,Very High,0


## Select Eligible Loans

Only borrowers that passed the previous decision engine are considered for portfolio construction.

Eligible borrowers include:

- Approve
- Manual Review

Rejected applications are excluded from portfolio optimization.

In [10]:
eligible_loans = df[df["loan_decision"].isin(["Approve","Manual Review"])].copy()
eligible_loans["loan_decision"].value_counts()

loan_decision
Manual Review    8000
Approve          3799
Name: count, dtype: int64

In [11]:
eligible_loans = eligible_loans.sort_values(by='risk_adjusted_return', ascending=False)
eligible_loans['portfolio_rank'] = range(1, len(eligible_loans) + 1)

In [12]:
eligible_loans[['portfolio_rank',"loan_decision","pd_score","expected_loss","expected_interest_income",
    "risk_adjusted_return"]].head()

,portfolio_rank,loan_decision,pd_score,expected_loss,expected_interest_income,risk_adjusted_return
18722,1,Approve,0.096850,1894.716335,4242.0,2347.283665
38246,2,Approve,0.126078,2466.501270,4546.5,2079.998730
39601,3,Approve,0.100250,1737.079032,3757.2,2020.120968
21374,4,Approve,0.162944,3187.733785,4931.5,1743.766215
11429,5,Approve,0.052854,982.389058,2621.5,1639.110942


## Select Optimized Portfolio

The top-ranked borrowers are selected to create the optimized lending portfolio.

For demonstration purposes, the portfolio size is fixed at 5,000 loans.

This simulates a lending institution selecting the most attractive applications under limited lending capacity.

In [13]:
size=5000
optimized_portfolio = eligible_loans.head(size).copy()
print(f'Optimized Portfolio Shape {optimized_portfolio.shape}')

Optimized Portfolio Shape (5000, 154)


## Portfolio Performance Summary

A reusable function is created to summarize key portfolio metrics.

The summary includes:

- borrower count
- actual default rate
- average PD
- average LGD
- total and average EAD
- expected loss
- expected interest income
- risk-adjusted return

This enables comparison between different lending portfolios.

In [17]:
def portfolio_summary(data, portfolio_name):
    return {
        'portfolio':portfolio_name,
        'borrower_count': len(data),
        'actual_default_rate': data['actual_default_flag'].mean(),
        'avg_pd': data['pd_score'].mean(),
        'avg_lgd': data['lgd_estimate'].mean(),
        'total_ead': data['ead'].sum(),
        'avg_ead' : data['ead'].mean(),
        'total_expected_loss': data['expected_loss'].sum(),
        'avg_expected_loss' : data['expected_loss'].mean(),
        'total_expected_interest_income' : data['expected_interest_income'].sum(),
        'avg_expected_interest_income' : data['expected_interest_income'].mean(),
        'total_risk_adjusted_return' : data['risk_adjusted_return'].sum(),
        'avg_risk_adjusted_return' : data['risk_adjusted_return'].mean()
             }

## Compare Eligible Portfolio vs Optimized Portfolio

The optimized portfolio is compared against the full eligible portfolio.

The comparison evaluates whether portfolio optimization improves:

- default rate
- expected loss
- profitability
- risk-adjusted return

This demonstrates the business value of risk-based portfolio selection.

In [20]:
portfolio_comparision = pd.DataFrame([portfolio_summary(eligible_loans, 'Eligible Portfolio'), 
                                     portfolio_summary(optimized_portfolio, 'Optimized Portfolio')])
portfolio_comparision

,portfolio,borrower_count,actual_default_rate,avg_pd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_expected_interest_income,avg_expected_interest_income,total_risk_adjusted_return,avg_risk_adjusted_return
0,Eligible Portfolio,11799,0.137469,0.326267,0.581394,160863325,13633.640563,3.105600e+07,2632.087337,1.896885e+07,1607.665941,-1.208715e+07,-1024.421396
1,Optimized Portfolio,5000,0.076800,0.192400,0.561357,56814975,11362.995000,4.634156e+06,926.831193,5.074427e+06,1014.885305,4.402706e+05,88.054113


## Optimized Porfolio By Risk Band

In [25]:
optimized_by_risk_band = optimized_portfolio.groupby('grade').agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
        total_expected_interest_income = ('expected_interest_income','sum'),
        avg_expected_interest_income = ('expected_interest_income','mean'),
    
).reset_index()

optimized_by_risk_band

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


,grade,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return,total_expected_interest_income,avg_expected_interest_income
0,A,2513,0.031834,1334.543177,0.531056,35624200,14175.964982,1.815813e+06,722.567877,617943.822983,245.898855,2.433757e+06,968.466732
1,B,1335,0.081648,746.200740,0.558952,15159125,11355.149813,1.583937e+06,1186.469934,105423.902803,78.969216,1.689361e+06,1265.439150
2,C,713,0.162693,435.992830,0.611491,4129675,5791.970547,7.666164e+05,1075.198298,-172495.534135,-241.929220,5.941209e+05,833.269078
3,D,332,0.189759,214.807956,0.647012,1481850,4463.403614,3.524356e+05,1061.553152,-87886.113808,-264.717210,2.645495e+05,796.835941
4,E,75,0.106667,52.173624,0.695648,292100,3894.666667,7.852579e+04,1047.010561,-17057.354606,-227.431395,6.146844e+04,819.579167
5,F,29,0.241379,20.766758,0.716095,89225,3076.724138,2.677086e+04,923.133005,-5520.454650,-190.360505,2.125040e+04,732.772500
6,G,3,0.333333,2.299257,0.766419,38800,12933.333333,1.005685e+04,3352.281883,-137.705650,-45.901883,9.919140e+03,3306.380000


## Portfolio Analysis by Loan Grade

The optimized portfolio is summarized across loan grades.

The analysis helps understand:

- borrower distribution
- default rates
- expected losses
- profitability
- exposure

Loan grades represent borrower credit quality and are commonly used by financial institutions.

In [21]:
optimized_by_grade = optimized_portfolio.groupby('grade').agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
        total_expected_interest_income = ('expected_interest_income','sum'),
        avg_expected_interest_income = ('expected_interest_income','mean'),
    
).reset_index()

optimized_by_grade

,grade,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return,total_expected_interest_income,avg_expected_interest_income
0,A,2513,0.031834,1334.543177,0.531056,35624200,14175.964982,1.815813e+06,722.567877,617943.822983,245.898855,2.433757e+06,968.466732
1,B,1335,0.081648,746.200740,0.558952,15159125,11355.149813,1.583937e+06,1186.469934,105423.902803,78.969216,1.689361e+06,1265.439150
2,C,713,0.162693,435.992830,0.611491,4129675,5791.970547,7.666164e+05,1075.198298,-172495.534135,-241.929220,5.941209e+05,833.269078
3,D,332,0.189759,214.807956,0.647012,1481850,4463.403614,3.524356e+05,1061.553152,-87886.113808,-264.717210,2.645495e+05,796.835941
4,E,75,0.106667,52.173624,0.695648,292100,3894.666667,7.852579e+04,1047.010561,-17057.354606,-227.431395,6.146844e+04,819.579167
5,F,29,0.241379,20.766758,0.716095,89225,3076.724138,2.677086e+04,923.133005,-5520.454650,-190.360505,2.125040e+04,732.772500
6,G,3,0.333333,2.299257,0.766419,38800,12933.333333,1.005685e+04,3352.281883,-137.705650,-45.901883,9.919140e+03,3306.380000


## Portfolio Analysis by Loan Purpose

The optimized portfolio is also analyzed by loan purpose.

Examples include:

- debt consolidation
- credit card
- home improvement
- medical
- small business
- vacation

This helps identify which loan purposes contribute the strongest risk-adjusted portfolio performance.

In [22]:
optimized_by_purpose = optimized_portfolio.groupby('purpose').agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
        total_expected_interest_income = ('expected_interest_income','sum'),
        avg_expected_interest_income = ('expected_interest_income','mean'),
    
).reset_index()

optimized_by_purpose

,purpose,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return,total_expected_interest_income,avg_expected_interest_income
0,car,90,0.077778,50.326893,0.559188,551900,6132.222222,5.374968e+04,597.218674,-4456.623129,-49.518035,4.929306e+04,547.700639
1,credit_card,1276,0.051724,699.438650,0.548149,17372525,13614.831505,1.218831e+06,955.196749,216013.980560,169.289953,1.434845e+06,1124.486703
2,debt_consolidation,2349,0.077480,1313.724202,0.559270,29304900,12475.478927,2.415255e+06,1028.205769,250909.638070,106.815512,2.666165e+06,1135.021281
3,educational,4,0.250000,2.260450,0.565113,24400,6100.000000,2.793228e+03,698.307051,-67.208206,-16.802051,2.726020e+03,681.505000
4,home_improvement,448,0.062500,249.452965,0.556815,4735575,10570.479911,3.402476e+05,759.481140,54934.609486,122.621896,3.951822e+05,882.103036
5,house,24,0.083333,13.698100,0.570754,327400,13641.666667,2.921296e+04,1217.206666,1777.450005,74.060417,3.099041e+04,1291.267083
6,major_purchase,175,0.085714,98.797105,0.564555,1230525,7031.571429,1.062049e+05,606.885390,954.009279,5.451482,1.071590e+05,612.336871
7,medical,86,0.186047,50.945923,0.592394,443825,5160.755814,5.974059e+04,694.657979,-9709.436165,-112.900421,5.003115e+04,581.757558
8,moving,59,0.186441,35.684312,0.604819,304175,5155.508475,4.730616e+04,801.799355,-9980.684454,-169.164143,3.732548e+04,632.635212
9,other,367,0.108992,219.664576,0.598541,1853075,5049.250681,2.637062e+05,718.545501,-42558.793832,-115.964016,2.211474e+05,602.581485


## Save Portfolio Outputs

The optimized portfolio and portfolio summaries are saved for downstream reporting and business analysis.

Saved outputs include:

- optimized borrower portfolio
- portfolio comparison
- portfolio summary by loan grade
- portfolio summary by loan purpose

These outputs can be used to support lending strategy and portfolio management decisions.

In [27]:
optimized_portfolio.to_pickle("../data/optimized_portfolio.pkl")

portfolio_comparision.to_csv("../data/portfolio_comparison.csv", index=False)
optimized_by_risk_band.to_csv("../data/optimized_portfolio_by_risk_band.csv", index=False)
optimized_by_grade.to_csv("../data/optimized_portfolio_by_grade.csv", index=False)
optimized_by_purpose.to_csv("../data/optimized_portfolio_by_purpose.csv", index=False)

# Conclusion

This notebook demonstrates how machine learning credit risk outputs can be transformed into a practical portfolio optimization framework.

Key steps completed:

1. Imported borrower-level decision engine output.
2. Selected eligible borrowers.
3. Ranked borrowers using risk-adjusted return.
4. Constructed an optimized lending portfolio.
5. Compared the optimized portfolio against the full eligible portfolio.
6. Analyzed portfolio performance by loan grade and loan purpose.
7. Saved portfolio-level summaries for reporting.

Key business insight:

Instead of approving every eligible borrower, financial institutions can prioritize applicants with stronger expected profitability and lower credit risk. This allows lenders to reduce expected losses while improving overall portfolio performance.

This notebook completes the end-to-end Lending Club credit risk pipeline by demonstrating how predictive models can support real-world lending decisions and portfolio optimization.